In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


BRONZE_SCHEMA = "football_data_platform.bronze"
SILVER_SCHEMA = "football_data_platform.silver"
SILVER_TABLE = f"{SILVER_SCHEMA}.team_data"


def clean_strings(df: DataFrame) -> DataFrame:
    """Trim string columns and convert empty strings to NULL."""

    for column, data_type in df.dtypes:
        if data_type == "string":
            df = df.withColumn(
                column,
                F.when(
                    F.trim(F.col(f"`{column}`")) == "",
                    None
                ).otherwise(
                    F.trim(F.col(f"`{column}`"))
                )
            )

    return df


# ---------------------------------------------------------
# 1. Read Bronze table
# ---------------------------------------------------------

table_name = f"{BRONZE_SCHEMA}.api_team_data"

if not spark.catalog.tableExists(table_name):
    raise ValueError(f"Table does not exist: {table_name}")

df = spark.table(table_name)
df = clean_strings(df)


# ---------------------------------------------------------
# 2. Select and standardize columns
# ---------------------------------------------------------

df_silver = (
    df
    .select(
        # Team
        F.col("data.team.id").cast("long").alias("team_id"),
        F.col("data.team.name").cast("string").alias("team_name"),
        F.col("data.team.code").cast("string").alias("team_code"),
        F.col("data.team.country").cast("string").alias("team_country"),
        F.col("data.team.founded").cast("int").alias("team_founded"),
        F.col("data.team.national").cast("boolean").alias("team_national"),

        # Venue
        F.col("data.venue.id").cast("long").alias("venue_id"),
        F.col("data.venue.name").cast("string").alias("venue_name"),
        F.col("data.venue.address").cast("string").alias("venue_address"),
        F.col("data.venue.city").cast("string").alias("venue_city"),
        F.col("data.venue.capacity").cast("int").alias("venue_capacity"),
        F.col("data.venue.surface").cast("string").alias("venue_surface"),

        # Context
        F.col("league_id").cast("long").alias("league_id"),
        F.col("league_name").cast("string").alias("league_name"),
        F.col("season").cast("int").alias("season"),

        # Bronze metadata
        F.col("_ingested_at")
            .cast("timestamp")
            .alias("_bronze_ingested_at"),

        F.col("_source_file")
            .cast("string")
            .alias("_bronze_source_file"),

        F.col("_source_system")
            .cast("string")
            .alias("_source_system")
    )
)


# ---------------------------------------------------------
# 3. Remove duplicates
# ---------------------------------------------------------

duplicate_columns = [
    "team_id",
    "league_id",
    "season"
]

duplicate_count = (
    df_silver
    .groupBy(duplicate_columns)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicate groups found: {duplicate_count}")

if duplicate_count > 0:
    df_silver = df_silver.dropDuplicates(duplicate_columns)
    print("Duplicates removed.")
else:
    print("No duplicates found. No changes made.")


# ---------------------------------------------------------
# 4. Add Silver metadata
# ---------------------------------------------------------

df_silver = df_silver.withColumn(
    "_silver_processed_at",
    F.current_timestamp()
)


# ---------------------------------------------------------
# 5. Basic validation
# ---------------------------------------------------------

null_team_ids = df_silver.filter(
    F.col("team_id").isNull()
).count()

if null_team_ids > 0:
    print(f"WARNING: NULL team_id: {null_team_ids}")


null_team_names = df_silver.filter(
    F.col("team_name").isNull()
).count()

if null_team_names > 0:
    print(f"WARNING: NULL team_name: {null_team_names}")


print(f"Silver rows: {df_silver.count()}")


# ---------------------------------------------------------
# 6. Write Silver
# ---------------------------------------------------------

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"Created {SILVER_TABLE}")